# 🇯🇵 日本の出生数分析（2009年〜2024年）
### 母親の年齢層別出生数
*出典：人口動態統計 第2巻 出生*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings

warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Noto Sans JP'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. データの読み込みと準備

In [ ]:
FILE_PATH = '/Users/rodrigobravo/Desktop/Jupyter/Japan_Maternity_Work_Analysis/Natality_Japan/Yearly_born/All_years.xlsx'

raw = pd.read_excel(FILE_PATH, sheet_name='Data')

age_map = {'-14 years': '15歳未満', '50-': '50歳以上'}
age_labels_en = ['<15', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50+']
age_labels_jp = ['15歳未満', '15〜19歳', '20〜24歳', '25〜29歳', '30〜34歳', '35〜39歳', '40〜44歳', '45〜49歳', '50歳以上']
en_to_jp = dict(zip(age_labels_en, age_labels_jp))

age_map_full = {'-14 years': '15歳未満', '15-19': '15〜19歳', '20-24': '20〜24歳',
                '25-29': '25〜29歳', '30-34': '30〜34歳', '35-39': '35〜39歳',
                '40-44': '40〜44歳', '45-49': '45〜49歳', '50-': '50歳以上'}
raw['Age range'] = raw['Age range'].replace(age_map_full)

filtered = raw[raw['Age range'].isin(age_labels_jp)]

df = filtered.groupby(['Year', 'Age range'])['Born'].sum().unstack('Age range')[age_labels_jp]
df.index.name = '年'

urban = filtered[filtered['Area'] == 'Urban'].groupby(['Year', 'Age range'])['Born'].sum().unstack('Age range')[age_labels_jp]
rural = filtered[filtered['Area'] == 'Rural'].groupby(['Year', 'Age range'])['Born'].sum().unstack('Age range')[age_labels_jp]

print(f'データ読み込み完了: {FILE_PATH}')
print(f'{len(df)}年分  ·  {len(age_labels_jp)}つの年齢層')
df

## 2. 出生数の全体推移

In [ ]:
totals = df.sum(axis=1)
gensho = (totals.iloc[-1] - totals.iloc[0]) / totals.iloc[0] * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(totals.index, totals.values, alpha=0.15, color='#2E75B6')
ax.plot(totals.index, totals.values, color='#2E75B6', linewidth=2.5, marker='o', markersize=5)

for year in [totals.index[0], totals.index[len(totals)//2], totals.index[-1]]:
    ax.annotate(f'{totals[year]:,.0f}人',
                xy=(year, totals[year]), xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=9, fontweight='bold', color='#2E75B6')

ax.set_title(f'日本の総出生数  （全体減少率: {gensho:.1f}%）', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('年')
ax.set_ylabel('出生数')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e4:.0f}万人' if x >= 1e4 else f'{x:,.0f}'))
ax.set_xticks(df.index)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(f'{df.index[0]}年: {totals.iloc[0]:,.0f}人')
print(f'{df.index[-1]}年: {totals.iloc[-1]:,.0f}人')
print(f'減少数: {totals.iloc[0]-totals.iloc[-1]:,.0f}人 （{gensho:.1f}%）')

## 3. 年齢層別の出生数推移

In [ ]:
colors = {
    '15歳未満': '#d62728', '15〜19歳': '#ff7f0e', '20〜24歳': '#e377c2',
    '25〜29歳': '#2ca02c', '30〜34歳': '#1f77b4', '35〜39歳': '#9467bd',
    '40〜44歳': '#8c564b', '45〜49歳': '#bcbd22', '50歳以上': '#17becf',
}

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, age in enumerate(age_labels_jp):
    ax = axes[i]
    vals = df[age]
    pct = (vals.iloc[-1] - vals.iloc[0]) / vals.iloc[0] * 100
    color = colors[age]
    ax.fill_between(vals.index, vals.values, alpha=0.12, color=color)
    ax.plot(vals.index, vals.values, color=color, linewidth=2, marker='o', markersize=3)
    arrow = '▼' if pct < 0 else '▲'
    pct_color = '#cc0000' if pct < 0 else '#006600'
    ax.set_title(age, fontsize=11, fontweight='bold')
    ax.text(0.97, 0.95, f'{arrow} {abs(pct):.1f}%', transform=ax.transAxes,
            ha='right', va='top', fontsize=10, fontweight='bold', color=pct_color)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
    ax.set_xticks(vals.index[::3])
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('母親の年齢層別出生数 — 日本', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. 絶対変化と変化率

In [ ]:
abs_change = df.iloc[-1] - df.iloc[0]
pct_change = (abs_change / df.iloc[0] * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bar_colors = ['#cc2222' if v < 0 else '#2a7a2a' for v in abs_change.values]
bars = ax1.barh(age_labels_jp, abs_change.values, color=bar_colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, abs_change.values):
    x = val - 2000 if val < 0 else val + 500
    ax1.text(x, bar.get_y() + bar.get_height()/2, f'{val:+,.0f}人',
             va='center', ha='right' if val < 0 else 'left', fontsize=9, fontweight='bold')
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_title(f'出生数の絶対変化\n{df.index[0]}年 → {df.index[-1]}年', fontsize=11, fontweight='bold')
ax1.set_xlabel('出生数')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

bar_colors2 = ['#cc2222' if v < 0 else '#2a7a2a' for v in pct_change.values]
bars2 = ax2.barh(age_labels_jp, pct_change.values, color=bar_colors2, edgecolor='white', height=0.6)
for bar, val in zip(bars2, pct_change.values):
    x = val - 1 if val < 0 else val + 0.5
    ax2.text(x, bar.get_y() + bar.get_height()/2, f'{val:+.1f}%',
             va='center', ha='right' if val < 0 else 'left', fontsize=9, fontweight='bold')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title(f'出生数の変化率\n{df.index[0]}年 → {df.index[-1]}年', fontsize=11, fontweight='bold')
ax2.set_xlabel('変化率（%）')

plt.suptitle('年齢層別の影響 — 日本', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. 都市部 vs 農村部

In [ ]:
pct_df = df.div(df.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# グラフ1: 若年層（初期値・最終値付き）
ax = axes[0]
for age, color in [('15〜19歳', '#ff7f0e'), ('20〜24歳', '#e377c2'), ('25〜29歳', '#2ca02c')]:
    vals = df[age]
    ax.plot(df.index, vals, label=age, color=color, linewidth=2.5, marker='o', markersize=4)
    ax.annotate(f'{vals.iloc[0]:,.0f}人',
                xy=(vals.index[0], vals.iloc[0]),
                xytext=(-8, 6), textcoords='offset points',
                fontsize=7.5, fontweight='bold', color=color, ha='right')
    ax.annotate(f'{vals.iloc[-1]:,.0f}人',
                xy=(vals.index[-1], vals.iloc[-1]),
                xytext=(6, 0), textcoords='offset points',
                fontsize=7.5, fontweight='bold', color=color, ha='left')
ax.set_title('若年層（15〜29歳）の出生数減少', fontsize=11, fontweight='bold')
ax.set_ylabel('出生数')
ax.set_xlabel('年')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend(title='年齢層')
ax.set_xticks(df.index)
ax.tick_params(axis='x', rotation=45)
if 2020 in df.index:
    ax.axvline(2020, color='gray', linestyle='--', alpha=0.5)
    ax.text(2020.1, ax.get_ylim()[1]*0.95, 'COVID-19', fontsize=8, color='gray')

# グラフ2: 20〜24歳 都市部 vs 農村部（初期値・最終値付き）
ax2 = axes[1]
for series, label, color, marker in [
    (urban['20〜24歳'], '都市部', '#1f77b4', 'o'),
    (rural['20〜24歳'], '農村部', '#2ca02c', 's'),
]:
    ax2.plot(series.index, series.values, label=label, color=color, linewidth=2.5, marker=marker, markersize=4)
    ax2.annotate(f'{series.iloc[0]:,.0f}人',
                 xy=(series.index[0], series.iloc[0]),
                 xytext=(-8, 6), textcoords='offset points',
                 fontsize=7.5, fontweight='bold', color=color, ha='right')
    ax2.annotate(f'{series.iloc[-1]:,.0f}人',
                 xy=(series.index[-1], series.iloc[-1]),
                 xytext=(6, 0), textcoords='offset points',
                 fontsize=7.5, fontweight='bold', color=color, ha='left')
ax2.set_title('20〜24歳：都市部 vs 農村部', fontsize=11, fontweight='bold')
ax2.set_ylabel('出生数')
ax2.set_xlabel('年')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax2.legend()
ax2.set_xticks(df.index)
ax2.tick_params(axis='x', rotation=45)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('フォーカス：20〜24歳', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. フォーカス：20〜24歳

In [ ]:
pct_df = df.div(df.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for age, color in [('15〜19歳', '#ff7f0e'), ('20〜24歳', '#e377c2'), ('25〜29歳', '#2ca02c')]:
    ax.plot(df.index, df[age], label=age, color=color, linewidth=2.5, marker='o', markersize=4)
ax.set_title('若年層（15〜29歳）の出生数減少', fontsize=11, fontweight='bold')
ax.set_ylabel('出生数')
ax.set_xlabel('年')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend(title='年齢層')
ax.set_xticks(df.index)
ax.tick_params(axis='x', rotation=45)
if 2020 in df.index:
    ax.axvline(2020, color='gray', linestyle='--', alpha=0.5)
    ax.text(2020.1, ax.get_ylim()[1]*0.95, 'COVID-19', fontsize=8, color='gray')

ax2 = axes[1]
ax2.plot(urban.index, urban['20〜24歳'], label='都市部', color='#1f77b4', linewidth=2.5, marker='o', markersize=4)
ax2.plot(rural.index, rural['20〜24歳'], label='農村部', color='#2ca02c', linewidth=2.5, marker='s', markersize=4)
ax2.set_title('20〜24歳：都市部 vs 農村部', fontsize=11, fontweight='bold')
ax2.set_ylabel('出生数')
ax2.set_xlabel('年')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax2.legend()
ax2.set_xticks(df.index)
ax2.tick_params(axis='x', rotation=45)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('フォーカス：20〜24歳', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. 高齢出産化：晩産化は進んでいるか？

In [ ]:
midpoints = {'15歳未満': 14, '15〜19歳': 17, '20〜24歳': 22, '25〜29歳': 27,
             '30〜34歳': 32, '35〜39歳': 37, '40〜44歳': 42, '45〜49歳': 47, '50歳以上': 51}

weighted_age = pd.Series({
    year: sum(df.loc[year, age] * midpoints[age] for age in age_labels_jp) / df.loc[year].sum()
    for year in df.index
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(weighted_age.index, weighted_age.values, color='#7030A0', linewidth=2.5, marker='o', markersize=5)
ax1.fill_between(weighted_age.index, weighted_age.values, weighted_age.min()-0.1, alpha=0.1, color='#7030A0')
for year in [weighted_age.index[0], weighted_age.index[-1]]:
    ax1.annotate(f'{weighted_age[year]:.2f}歳',
                 xy=(year, weighted_age[year]), xytext=(0, 10), textcoords='offset points',
                 ha='center', fontsize=9, fontweight='bold', color='#7030A0')
ax1.set_title('母親の平均出産年齢（推定）', fontsize=11, fontweight='bold')
ax1.set_ylabel('平均年齢（歳）')
ax1.set_xlabel('年')
ax1.set_xticks(df.index)
ax1.tick_params(axis='x', rotation=45)

ax2.plot(df.index, pct_df['20〜24歳'], label='20〜24歳', color='#e377c2', linewidth=2.5, marker='o', markersize=4)
ax2.plot(df.index, pct_df['30〜34歳'], label='30〜34歳', color='#1f77b4', linewidth=2.5, marker='o', markersize=4)
ax2.plot(df.index, pct_df['35〜39歳'], label='35〜39歳', color='#9467bd', linewidth=2, marker='s', markersize=4, linestyle='--')
ax2.set_title('晩産化は進んでいるか？\n年齢層別の出生構成比の推移', fontsize=11, fontweight='bold')
ax2.set_ylabel('総出生数に占める割合（%）')
ax2.set_xlabel('年')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax2.legend(title='年齢層')
ax2.set_xticks(df.index)
ax2.tick_params(axis='x', rotation=45)

plt.suptitle('晩産化の進行 — 日本', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'{df.index[0]}年の平均出産年齢: {weighted_age.iloc[0]:.2f}歳')
print(f'{df.index[-1]}年の平均出産年齢: {weighted_age.iloc[-1]:.2f}歳')
print(f'晩産化の進行: +{weighted_age.iloc[-1]-weighted_age.iloc[0]:.2f}歳')

## 8. 年齢層別：婚内・婚外出生数の推移

In [ ]:
wedlock     = filtered.groupby(['Year', 'Age range'])['Born in wedlock'].sum().unstack('Age range')[age_labels_jp]
out_wedlock = filtered.groupby(['Year', 'Age range'])['Born out of wedlock'].sum().unstack('Age range')[age_labels_jp]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, age in enumerate(age_labels_jp):
    ax = axes[i]
    w  = wedlock[age]
    ow = out_wedlock[age]
    total = w + ow

    ax.stackplot(wedlock.index,
                 [w.values, ow.values],
                 labels=['婚内出生', '婚外出生'],
                 colors=[colors[age], '#dddddd'],
                 alpha=0.85)

    pct_ini = ow.iloc[0] / total.iloc[0] * 100
    pct_fin = ow.iloc[-1] / total.iloc[-1] * 100
    ax.set_title(age, fontsize=10, fontweight='bold')
    ax.text(0.97, 0.97, f'婚外: {pct_ini:.1f}% → {pct_fin:.1f}%',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=7.5, color='#555555',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
    ax.set_xticks(wedlock.index[::3])
    ax.tick_params(labelsize=7.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.suptitle('年齢層別 婚内・婚外出生数の推移 — 日本',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. 婚外出生割合の推移（年齢層別）

In [ ]:
# 年齢層別 婚外出生の割合推移
pct_out = (out_wedlock / (wedlock + out_wedlock) * 100).round(2)

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, age in enumerate(age_labels_jp):
    ax = axes[i]
    vals = pct_out[age]
    color = colors[age]

    ax.fill_between(vals.index, vals.values, alpha=0.15, color=color)
    ax.plot(vals.index, vals.values, color=color, linewidth=2.5, marker='o', markersize=4)

    ax.annotate(f'{vals.iloc[0]:.1f}%',
                xy=(vals.index[0], vals.iloc[0]),
                xytext=(6, 4), textcoords='offset points',
                fontsize=8, fontweight='bold', color=color)
    ax.annotate(f'{vals.iloc[-1]:.1f}%',
                xy=(vals.index[-1], vals.iloc[-1]),
                xytext=(-6, 4), textcoords='offset points',
                fontsize=8, fontweight='bold', color=color, ha='right')

    diff = vals.iloc[-1] - vals.iloc[0]
    arrow = '▲' if diff > 0 else '▼'
    arrow_color = '#cc0000' if diff > 0 else '#006600'
    ax.text(0.97, 0.08, f'{arrow} {abs(diff):.1f}pp',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, fontweight='bold', color=arrow_color)

    ax.set_title(age, fontsize=10, fontweight='bold')
    ax.set_ylabel('%')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
    ax.set_xticks(vals.index[::3])
    ax.tick_params(labelsize=7.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('年齢層別 婚外出生割合の推移 — 日本',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n婚外出生割合 — 初年度 vs 最終年度:')
for age in age_labels_jp:
    diff = pct_out[age].iloc[-1] - pct_out[age].iloc[0]
    arrow = '▲' if diff > 0 else '▼'
    print(f'  {age}: {pct_out[age].iloc[0]:.1f}% → {pct_out[age].iloc[-1]:.1f}%  ({arrow} {abs(diff):.1f}pp)')

## 10. エグゼクティブサマリー

In [ ]:
print('=' * 55)
print(f'   日本の出生数分析 {df.index[0]}年〜{df.index[-1]}年')
print('=' * 55)
print(f'\n📉 {df.index[0]}年の総出生数: {totals.iloc[0]:>10,.0f}人')
print(f'📉 {df.index[-1]}年の総出生数: {totals.iloc[-1]:>10,.0f}人')
print(f'   総減少数:          {totals.iloc[0]-totals.iloc[-1]:>10,.0f}人 （{gensho:.1f}%）')

print('\n🔴 減少率が最も大きい年齢層:')
for age in pct_change.sort_values().head(5).index:
    print(f'   {age}: {pct_change[age]:>+7.1f}%  （{abs_change[age]:>+8,.0f}人）')

print('\n🟢 増加している年齢層:')
rising = pct_change[pct_change > 0]
if len(rising):
    for age in rising.index:
        print(f'   {age}: {pct_change[age]:>+7.1f}%  （{abs_change[age]:>+8,.0f}人）')
else:
    print('   なし — すべての年齢層で減少')

print(f'\n⏩ 晩産化の状況:')
print(f'   {df.index[0]}年の平均出産年齢: {weighted_age.iloc[0]:.2f}歳')
print(f'   {df.index[-1]}年の平均出産年齢: {weighted_age.iloc[-1]:.2f}歳')
print(f'   平均で +{weighted_age.iloc[-1]-weighted_age.iloc[0]:.2f}歳 の晩産化が進行')

print('\n💡 20〜24歳についての考察:')
print('   若い女性が子どもを持ちたくないわけではなく、')
print('   経済的・文化的・教育的な要因により')
print('   出産の決断が30歳代以降に先送りされている。')
print('=' * 55)